# Stan Radon Random Restarts

Runs Stan mean-field ADVI random restarts for the PyMC-tutorial varying-intercept radon model.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/radon_example/pymc_tutorial")
STAN_FILE_NAME = "stan_radon_varying_intercept.stan"

RADON_RUN_DIR = Path(os.environ.get("SIMPLEVI_SCRATCH", Path.home() / "simplevi_scratch")) / "radon_cmdstan_runs"  # off-iCloud scratch for speed


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")


Notebook dir: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/radon_example/pymc_tutorial
Stan model: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/radon_example/pymc_tutorial/stan_radon_varying_intercept.stan
Python executable: /Users/madelynandersen/miniconda3/envs/stan3/bin/python
CmdStan path: /Users/madelynandersen/miniconda3/envs/stan3/bin/cmdstan


/Users/madelynandersen/miniconda3/envs/stan3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modulars.radon import (
    default_radon_plot_dims,
    load_best_references,
    load_radon_data,
    make_radon_unconstrained_param_names,
    print_reference_summary,
)

radon_data = load_radon_data()
county_names = radon_data["county_names"]
param_names = make_radon_unconstrained_param_names(county_names)
dim = len(param_names)
which_dims, which_labels = default_radon_plot_dims(param_names, county_names)

stan_data = {
    "N": int(len(radon_data["log_radon"])),
    "C": int(len(county_names)),
    "log_radon": np.asarray(radon_data["log_radon"], dtype=float),
    "floor": np.asarray(radon_data["floor"], dtype=float),
    "county": np.asarray(radon_data["county"], dtype=int) + 1,
}
PARAM_COLUMNS = (
    ["mu_a", "sigma_a"]
    + [f"alpha[{i}]" for i in range(1, len(county_names) + 1)]
    + ["beta", "sd_y"]
)
LOG_COLUMNS = ["sigma_a", "sd_y"]

print(f"N observations: {stan_data['N']}")
print(f"N counties: {stan_data['C']}")
print(f"Latent VI dimension: {dim}")
print(list(zip(which_dims, which_labels)))


N observations: 919
N counties: 85
Latent VI dimension: 89
[(0, 'mu_a'), (87, 'beta'), (27, 'alpha[HENNEPIN]'), (71, 'alpha[ST LOUIS]')]


In [3]:
from pathlib import Path

RUN_MODE = os.environ.get("SIMPLEVI_STAN_RUN_MODE", "full")  # "quick" or "full"
RUN_CONFIGS = {
    "quick": {"max_iters": 20, "n_restarts": 1, "track_every": 10, "parallel": False, "max_workers": None, "keep_outputs": True},
    "full": {"max_iters": 300_000, "n_restarts": 20, "track_every": 10, "parallel": True, "max_workers": None, "keep_outputs": False},
}

run_config = RUN_CONFIGS[RUN_MODE]
max_iters = int(os.environ.get("SIMPLEVI_STAN_MAX_ITERS", run_config["max_iters"]))
n_restarts = int(os.environ.get("SIMPLEVI_STAN_N_RESTARTS", run_config["n_restarts"]))
track_every = int(os.environ.get("SIMPLEVI_STAN_TRACK_EVERY", run_config["track_every"]))
parallel = bool(int(os.environ.get("SIMPLEVI_STAN_PARALLEL", int(run_config["parallel"]))))
max_workers_env = os.environ.get("SIMPLEVI_STAN_MAX_WORKERS")
max_workers = int(max_workers_env) if max_workers_env else run_config["max_workers"]
keep_outputs = bool(int(os.environ.get("SIMPLEVI_STAN_KEEP_OUTPUTS", int(run_config["keep_outputs"]))))

results_dir = Path(os.environ.get("SIMPLEVI_STAN_RESULTS_DIR", "results"))
results_dir.mkdir(exist_ok=True, parents=True)

run_config | {"parallel": parallel, "max_workers": max_workers, "results_dir": str(results_dir)}


{'max_iters': 300000,
 'n_restarts': 20,
 'track_every': 10,
 'parallel': True,
 'max_workers': None,
 'keep_outputs': False,
 'results_dir': 'results'}

In [ ]:
from modulars.stan_rr_test import run_stan_random_restarts, stan_result_by_scenario, FOUR_WAY_STAN_SCENARIOS

TOL_TARGET_ITERS = 100_000
five_way_scenarios = [dict(s) for s in FOUR_WAY_STAN_SCENARIOS]
for _scenario in five_way_scenarios:
    if _scenario["name"] == "grad100_tol_rel_obj1e-6":
        _scenario["max_iters_scale"] = TOL_TARGET_ITERS / max_iters
        _scenario["track_every_scale"] = 15.0
five_way_scenarios.append({
    "name": "grad100_tol_rel_obj1e-6_eta0.01",
    "mc_label": "100_tol1e-6_eta0.01",
    "cmd_args": ("grad_samples=100", "adapt", "engaged=0", "eta=0.01"),
    "tol_rel_obj": 1e-6,
    "max_iters_scale": TOL_TARGET_ITERS / max_iters,
    "track_every_scale": 15.0,
})

five_way_scenarios.append({
    "name": "tol_1e-6_default_rest",
    "mc_label": "tol_1e-6_default_rest",
    "cmd_args": (),
    "tol_rel_obj": 1e-6,
    "max_iters_scale": 75_000 / max_iters,
    "track_every_scale": 150 / track_every,
})
stan_result = run_stan_random_restarts(
    stan_file=STAN_FILE,
    data=stan_data,
    param_columns=PARAM_COLUMNS,
    log_columns=LOG_COLUMNS,
    output_dir=RADON_RUN_DIR,
    max_iters=max_iters,
    n_restarts=n_restarts,
    track_every=track_every,
    seed_offset=0,
    parallel=parallel,
    max_workers=max_workers,
    keep_outputs=keep_outputs,
    refresh=0,
    scenarios=five_way_scenarios,
)
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {RADON_RUN_DIR / 'stan_run_failures.csv'}")



Stan restart/scenario tasks:   0%|          | 0/50 [00:00<?, ?it/s]


Stan restart/scenario tasks:   0%|          | 0/50 [00:00<?, ?it/s]


Stan restart/scenario tasks:   2%|▏         | 1/50 [04:58<4:04:06, 298.91s/it]


Stan restart/scenario tasks:   4%|▍         | 2/50 [05:04<1:40:51, 126.08s/it]


Stan restart/scenario tasks:   6%|▌         | 3/50 [05:17<58:29, 74.67s/it]   


Stan restart/scenario tasks:   8%|▊         | 4/50 [05:34<39:42, 51.80s/it]

... (44 outputs left out)


Stan restart/scenario tasks:  98%|█████████▊| 49/50 [34:17:30<42:17, 2537.32s/it]  


Stan restart/scenario tasks: 100%|██████████| 50/50 [34:19:16<00:00, 1807.95s/it]


Stan restart/scenario tasks: 100%|██████████| 50/50 [34:19:16<00:00, 2471.13s/it]


Stan restart/scenario tasks:   0%|          | 0/50 [34:19:16<?, ?it/s]

In [5]:
import pickle

STAN_RESULT_PKL = results_dir / "stan_result.pkl"
with open(STAN_RESULT_PKL, "wb") as f:
    pickle.dump(stan_result, f)
print(f"Saved Stan results to {STAN_RESULT_PKL}")

Saved Stan results to results/stan_result.pkl


In [6]:
import pickle

STAN_RESULT_PKL = results_dir / "stan_result.pkl"
with open(STAN_RESULT_PKL, "rb") as f:
    stan_result = pickle.load(f)
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")
print(f"Loaded Stan results from {STAN_RESULT_PKL}")

Loaded Stan results from results/stan_result.pkl


In [7]:
by_scenario = stan_result_by_scenario(stan_result)

default_means, default_stds, default_iterations = by_scenario["default"]
grad100_means, grad100_stds, grad100_iterations = by_scenario["grad100_default_rest"]
adapt_off_means, adapt_off_stds, adapt_off_iterations = by_scenario["grad100_adapt_off"]
tol_means, tol_stds, tol_iterations = by_scenario["grad100_tol_rel_obj1e-6"]
eta_means, eta_stds, eta_iterations = by_scenario["grad100_tol_rel_obj1e-6_eta0.01"]

import numpy as np

# "matplotlib" (default, static, savable via plt.savefig) or "plotly"
# (interactive: zoom/pan/hover, shown inline). Flip this and rerun the
# plotting cell below to switch how every plot in this notebook renders.
PLOT_BACKEND = "plotly"


In [7]:
print_reference_summary()
from modulars.plot_rr import plot_a_few_trajectories_grid_1d
reference_summary, best_reference, reference_means, reference_stds = load_best_references()


def downsample_and_extend(arr, factor, target_len):
    """Downsample along the last (iteration) axis by `factor`, then hold
    the last value flat out to `target_len` points -- a valid projection
    once a run has converged and stopped changing, used here to compare
    the (short) default run against the (long) tolerance-ablation run on
    the latter's own iteration grid.
    """
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled


# non-tolerance comparisons: these three scenarios share the default
# scenario's own iteration grid (all three use the same max_iters/track_every).
for name, means, stds, scen_iterations in (
    ("grad100_default_rest", grad100_means, grad100_stds, grad100_iterations),
    ("grad100_adapt_off", adapt_off_means, adapt_off_stds, adapt_off_iterations),
):
    plot_a_few_trajectories_grid_1d(
        default_means, default_stds, means, stds,
        dims=which_dims, labels=which_labels, title_prefix=f"Stan {name} ",
        reference_means=reference_means, reference_stds=reference_stds,
        k=2, x=scen_iterations, backend=PLOT_BACKEND,
        single_label="default", multi_label=name,
    )

# tolerance comparison: the tolerance scenario runs 2x longer (and is tracked
# 10x coarser) than the default run, so we downsample the default run's
# trajectory by 10x to match the tolerance scenario's tracking step, then
# project its last (converged) value flat out to match tol_iterations'
# length (it does not change once converged).
projected_default_means = downsample_and_extend(default_means, 10, len(tol_iterations))
projected_default_stds = downsample_and_extend(default_stds, 10, len(tol_iterations))
plot_a_few_trajectories_grid_1d(
    projected_default_means, projected_default_stds, tol_means, tol_stds,
    dims=which_dims, labels=which_labels, title_prefix="Stan grad100_tol_rel_obj1e-6 ",
    reference_means=reference_means, reference_stds=reference_stds,
    k=2, x=tol_iterations, backend=PLOT_BACKEND,
    single_label="default", multi_label="grad100_tol_rel_obj1e-6",
)

projected_default_means_eta = downsample_and_extend(default_means, 10, len(eta_iterations))
projected_default_stds_eta = downsample_and_extend(default_stds, 10, len(eta_iterations))
plot_a_few_trajectories_grid_1d(
    projected_default_means_eta, projected_default_stds_eta, eta_means, eta_stds,
    dims=which_dims, labels=which_labels, title_prefix="Stan grad100_tol_rel_obj1e-6_eta0.01 ",
    reference_means=reference_means, reference_stds=reference_stds,
    k=2, x=eta_iterations, backend=PLOT_BACKEND,
    single_label="default", multi_label="grad100_tol_rel_obj1e-6_eta0.01",
)

PyMC tutorial exact reported summaries:
{'beta': {'mean': -0.664, 'sd': 0.069}}

Reference notes:
- pymc_tutorial: The notebooks implement the PyMC tutorial's centered varying-intercept model: county intercepts, a shared floor effect, and Exponential(1) priors on the positive scale parameters.
- best_advi_reference: Parameters not reported directly by the PyMC tutorial are compared against the shared saved-output VI reference in best_reference_values.csv. The reference script selects the saved 100-MC variational restart with the largest NumPyro-computed fixed-guide ELBO.
- price_nero_gelman_1996: The peer-reviewed Price/Nero/Gelman Health Physics article uses a different county-geometric-mean model on adjusted Bq/m^3 radon values; those comparisons live in single_MC/radon_example/price_nero_gelman_1996.


In [8]:
import pickle

from modulars.radon import plot_radon_mean_band_grid_single
from modulars.stan_rr_test import stan_result_by_scenario

with open("tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

for name, means, stds, scen_iterations in (
    ("default", default_means, default_stds, default_iterations),
    ("grad100_default_rest", grad100_means, grad100_stds, grad100_iterations),
    ("grad100_adapt_off", adapt_off_means, adapt_off_stds, adapt_off_iterations),
    ("grad100_tol_rel_obj1e-6", tol_means, tol_stds, tol_iterations),
    ("grad100_tol_rel_obj1e-6_eta0.01", eta_means, eta_stds, eta_iterations),
    ("tol_1e-6_default_rest", td_means, td_stds, td_iterations),
):
    plot_radon_mean_band_grid_single(
        means, stds, scen_iterations, title=f"Stan {name} — posterior mean/std across restarts",
        which_dims=which_dims, which_labels=which_labels,
        reference_means=reference_means, reference_stds=reference_stds,
    )


In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modulars").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.radon import (
    default_radon_plot_dims,
    load_best_references,
    load_radon_data,
    make_radon_unconstrained_param_names,
)

NB_DIR = REPO_ROOT / "single_MC/radon_example/pymc_tutorial"
import os
os.chdir(NB_DIR)

radon_data = load_radon_data()
county_names = radon_data["county_names"]
param_names = make_radon_unconstrained_param_names(county_names)
which_dims, which_labels = default_radon_plot_dims(param_names, county_names)

reference_summary, best_reference, reference_means, reference_stds = load_best_references()

with open(NB_DIR / "results/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
default_means, default_stds, default_iterations = _by_scenario["default"]
tol_means, tol_stds, tol_iterations = _by_scenario["grad100_tol_rel_obj1e-6"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]

n_td_failures = len(_td_result["failures"])
print(f"tol_1e-6_default_rest: {td_means.shape[0]} restarts, {len(td_iterations)} tracked points, "
      f"final iter={int(td_iterations[-1])}")
print(f"failures: {n_td_failures}")


def downsample_and_extend(arr, factor, target_len):
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled

tol_1e-6_default_rest: 10 restarts, 501 tracked points, final iter=75000
failures: 0


In [1]:
import pickle
import sys
from pathlib import Path
import numpy as np

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modulars").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modulars.stan_rr_test import stan_result_by_scenario
from modulars.plot_rr import plot_a_few_trajectories_grid_1d
from modulars.radon import default_radon_plot_dims, load_best_references, load_radon_data, make_radon_unconstrained_param_names

NB_DIR = REPO_ROOT / "single_MC/radon_example/pymc_tutorial"
import os
os.chdir(NB_DIR)

radon_data = load_radon_data()
county_names = radon_data["county_names"]
param_names = make_radon_unconstrained_param_names(county_names)
which_dims, which_labels = default_radon_plot_dims(param_names, county_names)

reference_summary, best_reference, reference_means, reference_stds = load_best_references()

with open(NB_DIR / "results/stan_result.pkl", "rb") as f:
    _main_result = pickle.load(f)
_by_scenario = stan_result_by_scenario(_main_result)
default_means, default_stds, default_iterations = _by_scenario["default"]

with open(NB_DIR / "tol_1e-6_default_rest_result.pkl", "rb") as f:
    _td_result = pickle.load(f)
_td_by_scenario = stan_result_by_scenario(_td_result)
td_means, td_stds, td_iterations = _td_by_scenario["tol_1e-6_default_rest"]


def downsample_and_extend(arr, factor, target_len):
    downsampled = arr[..., ::factor]
    n_have = downsampled.shape[-1]
    if n_have < target_len:
        pad = np.repeat(downsampled[..., -1:], target_len - n_have, axis=-1)
        downsampled = np.concatenate([downsampled, pad], axis=-1)
    return downsampled

projected_default_means = downsample_and_extend(default_means, 15, len(td_iterations))
projected_default_stds = downsample_and_extend(default_stds, 15, len(td_iterations))
plot_a_few_trajectories_grid_1d(
    projected_default_means, projected_default_stds, td_means, td_stds,
    dims=which_dims, labels=which_labels, title_prefix="Stan tol_1e-6_default_rest ",
    reference_means=reference_means, reference_stds=reference_stds,
    k=2, x=td_iterations, backend="plotly",
    single_label="Default (some runs)", multi_label="tol_1e-6_default_rest (some runs)",
)

In [2]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

LINE_ALPHA = 0.65
n_rows = len(which_dims)
subplot_titles = []
for lab in which_labels:
    subplot_titles.append(f"{lab}: posterior mean")
    subplot_titles.append(f"{lab}: posterior std")

fig = make_subplots(rows=n_rows, cols=2, vertical_spacing=0.4 / n_rows, horizontal_spacing=0.12,
                     subplot_titles=subplot_titles)

for row_idx, d in enumerate(which_dims):
    r = row_idx + 1
    show_legend = row_idx == 0
    lab = which_labels[row_idx]
    ref_mean = reference_means[lab][0][0] if lab in reference_means else None
    ref_std = reference_stds[lab][0][0] if lab in reference_stds else None

    fig.add_trace(go.Scatter(x=tol_iterations, y=np.nanmean(tol_means[:, :, d], axis=0), mode="lines",
                              line=dict(color="rgb(148,103,189)", width=2), opacity=LINE_ALPHA,
                              name="grad100_tol_rel_obj1e-6", legendgroup="tol", showlegend=show_legend), row=r, col=1)
    fig.add_trace(go.Scatter(x=td_iterations, y=np.nanmean(td_means[:, :, d], axis=0), mode="lines",
                              line=dict(color="rgb(31,119,180)", width=2), opacity=LINE_ALPHA,
                              name="tol_1e-6_default_rest", legendgroup="default_rest", showlegend=show_legend), row=r, col=1)
    if ref_mean is not None:
        fig.add_hline(y=ref_mean, line=dict(color="red", dash="dash"), row=r, col=1)

    fig.add_trace(go.Scatter(x=tol_iterations, y=np.nanmean(tol_stds[:, :, d], axis=0), mode="lines",
                              line=dict(color="rgb(148,103,189)", width=2), opacity=LINE_ALPHA,
                              name="grad100_tol_rel_obj1e-6", legendgroup="tol", showlegend=False), row=r, col=2)
    fig.add_trace(go.Scatter(x=td_iterations, y=np.nanmean(td_stds[:, :, d], axis=0), mode="lines",
                              line=dict(color="rgb(31,119,180)", width=2), opacity=LINE_ALPHA,
                              name="tol_1e-6_default_rest", legendgroup="default_rest", showlegend=False), row=r, col=2)
    if ref_std is not None:
        fig.add_hline(y=ref_std, line=dict(color="red", dash="dash"), row=r, col=2)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dash"),
                          name="best variational approximation"), row=1, col=1)

for r in range(1, n_rows + 1):
    fig.update_xaxes(title_text="iteration", range=[0, 75_000], row=r, col=1)
    fig.update_xaxes(title_text="iteration", range=[0, 75_000], row=r, col=2)

fig.update_layout(height=300 * n_rows, width=1000, title_text="Stan tol_1e-6_default_rest vs grad100_tol_rel_obj1e-6")
fig.show()

In [4]:
CHECKPOINTS = [50_000, 100_000, 500_000, 1_000_000]

with open(NB_DIR / "default_checkpoint_results.pkl", "rb") as f:
    _cp_results = pickle.load(f)

def pct_error(setup, best):
    return (setup - best) / best * 100.0

def zscore(setup_mean, best_mean, best_std):
    return (setup_mean - best_mean) / best_std

best_mean_full = np.array([reference_means[lab][0][0] if lab in reference_means else np.nan for lab in param_names])
best_std_full = np.array([reference_stds[lab][0][0] if lab in reference_stds else np.nan for lab in param_names])

print("===== radon (89 dims, summarized): percent-error / z-score vs best variational approximation =====")

rows = []
for cp in CHECKPOINTS:
    r = _cp_results[cp]
    m = np.asarray(r["means_by_scenario"]["default_single"][0, 0, :], dtype=float)
    s = np.asarray(r["stds_by_scenario"]["default_single"][0, 0, :], dtype=float)
    rows.append((f"default, single restart to {cp}", m, s))

for label, scen_name in [
    ("grad100_default_rest (final)", "grad100_default_rest"),
    ("grad100_adapt_off (final)", "grad100_adapt_off"),
    ("grad100_tol_rel_obj1e-6 (final)", "grad100_tol_rel_obj1e-6"),
    ("grad100_tol_rel_obj1e-6_eta0.01 (final)", "grad100_tol_rel_obj1e-6_eta0.01"),
    ("tol_1e-6_default_rest (final)", "tol_1e-6_default_rest"),
]:
    if scen_name == "tol_1e-6_default_rest":
        means, stds, iters = td_means, td_stds, td_iterations
    else:
        means, stds, iters = _by_scenario[scen_name]
    m = np.nanmean(means[:, -1, :], axis=0)
    s = np.nanmean(stds[:, -1, :], axis=0)
    rows.append((f"{label} (iter={int(iters[-1])})", m, s))

for row_name, m, s in rows:
    pe_mean = pct_error(m, best_mean_full)
    pe_std = pct_error(s, best_std_full)
    z = zscore(m, best_mean_full, best_std_full)
    print(
        f"  {row_name:45s}  "
        f"pct_err_mean[mean/median/max|.|]={np.nanmean(pe_mean):8.2f}/{np.nanmedian(pe_mean):8.2f}/{np.nanmax(np.abs(pe_mean)):8.2f}  "
        f"pct_err_std[mean/median/max|.|]={np.nanmean(pe_std):8.2f}/{np.nanmedian(pe_std):8.2f}/{np.nanmax(np.abs(pe_std)):8.2f}  "
        f"z[mean/median/max|.|]={np.nanmean(z):7.3f}/{np.nanmedian(z):7.3f}/{np.nanmax(np.abs(z)):7.3f}"
    )

print("\n----- representative dims (same ones used throughout this notebook) -----")
for row_name, m, s in rows:
    for d, lab in zip(which_dims, which_labels):
        pe_mean_d = pct_error(m[d], best_mean_full[d])
        pe_std_d = pct_error(s[d], best_std_full[d])
        z_d = zscore(m[d], best_mean_full[d], best_std_full[d])
        print(f"  {row_name:45s} {lab:14s}  pct_err_mean={pe_mean_d:9.3f}%  pct_err_std={pe_std_d:9.3f}%  z={z_d:7.3f}")

===== radon (89 dims, summarized): percent-error / z-score vs best variational approximation =====
  default, single restart to 50000               pct_err_mean[mean/median/max|.|]=   -0.26/    0.50/   14.52  pct_err_std[mean/median/max|.|]=    9.86/    9.54/   34.84  z[mean/median/max|.|]=  0.002/  0.042/  1.142
  default, single restart to 100000              pct_err_mean[mean/median/max|.|]=   -0.26/    0.50/   14.52  pct_err_std[mean/median/max|.|]=    9.86/    9.54/   34.84  z[mean/median/max|.|]=  0.002/  0.042/  1.142
  default, single restart to 500000              pct_err_mean[mean/median/max|.|]=   -0.26/    0.50/   14.52  pct_err_std[mean/median/max|.|]=    9.86/    9.54/   34.84  z[mean/median/max|.|]=  0.002/  0.042/  1.142
  default, single restart to 1000000             pct_err_mean[mean/median/max|.|]=   -0.26/    0.50/   14.52  pct_err_std[mean/median/max|.|]=    9.86/    9.54/   34.84  z[mean/median/max|.|]=  0.002/  0.042/  1.142
  grad100_default_rest (final) (iter=

In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

default_trace_mean = default_means[0]
default_trace_std = default_stds[0]

cp_means_full = np.array([np.asarray(_cp_results[cp]["means_by_scenario"]["default_single"][0, 0, :], dtype=float) for cp in CHECKPOINTS])
cp_stds_full = np.array([np.asarray(_cp_results[cp]["stds_by_scenario"]["default_single"][0, 0, :], dtype=float) for cp in CHECKPOINTS])
cp_iters = np.array(CHECKPOINTS)

n_rows = len(which_dims)
subplot_titles = []
for lab in which_labels:
    subplot_titles.append(f"{lab}: posterior mean")
    subplot_titles.append(f"{lab}: posterior std")

fig = make_subplots(rows=n_rows, cols=2, vertical_spacing=0.4 / n_rows, horizontal_spacing=0.12,
                     subplot_titles=subplot_titles)

for row_idx, d in enumerate(which_dims):
    r = row_idx + 1
    show_legend = row_idx == 0
    lab = which_labels[row_idx]
    ref_mean = reference_means[lab][0][0] if lab in reference_means else None
    ref_std = reference_stds[lab][0][0] if lab in reference_stds else None

    fig.add_trace(go.Scatter(x=default_iterations, y=default_trace_mean[:, d], mode="lines",
                              line=dict(color="rgb(31,119,180)", width=1.5),
                              name="Single restart (same seed)", legendgroup="line", showlegend=show_legend), row=r, col=1)
    fig.add_trace(go.Scatter(x=cp_iters, y=cp_means_full[:, d], mode="markers", marker=dict(color="green", size=9),
                              name="Single extended run example", legendgroup="dots", showlegend=show_legend), row=r, col=1)
    if ref_mean is not None:
        fig.add_hline(y=ref_mean, line=dict(color="red", dash="dash"), row=r, col=1)

    fig.add_trace(go.Scatter(x=default_iterations, y=default_trace_std[:, d], mode="lines",
                              line=dict(color="rgb(31,119,180)", width=1.5),
                              name="Single restart (same seed)", legendgroup="line", showlegend=False), row=r, col=2)
    fig.add_trace(go.Scatter(x=cp_iters, y=cp_stds_full[:, d], mode="markers", marker=dict(color="green", size=9),
                              name="Single extended run example", legendgroup="dots", showlegend=False), row=r, col=2)
    if ref_std is not None:
        fig.add_hline(y=ref_std, line=dict(color="red", dash="dash"), row=r, col=2)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dash"),
                          name="best variational approximation"), row=1, col=1)

for r in range(1, n_rows + 1):
    fig.update_xaxes(type="log", title_text="iteration (log scale)", row=r, col=1)
    fig.update_xaxes(type="log", title_text="iteration (log scale)", row=r, col=2)

fig.update_layout(height=300 * n_rows, width=1000, title_text="radon — posterior mean/std, default settings out to long iteration horizons")
fig.show()